# Tutorial 03 · Outlier treatment: find them, decide, replace
**Data Mining · HTW Berlin**

An **outlier** is a value far away from the others. Some are **errors** (a typo, a crash saved as 0). Some are **real but rare** (a very strong person, a member who almost never comes). Python finds them. **You** decide what to do.

| Part | Data | What you do |
|---|---|---|
| 1 | gym machines: weight, sets and repetitions | see what real outliers do to the mean, the scale and a prediction (just run) |
| 2 | Kiez Gym bookings (from last week) | measure, decide and **replace** outliers yourself |
| 3 | Kiez Gym | does the treatment change a model? (just run) |
| 4 | credit risk: 32,581 real bank loans | one more example: the same steps on real data |

**How it works:** first an **Example**, then **Your turn** with almost the same code. Replace every `____` and run the cell. The answers are at the end of the notebook.

## 0 · Setup (just run, you do not need to read it)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
STACK = list(range(5, 135, 5))      # the weight stack of a machine: 5, 10, ..., 130 kg


def find_data(name):
    """Look for a data file in the usual folders."""
    from pathlib import Path
    for folder in [Path("data"), Path("../data"), Path(".")]:
        if (folder / name).exists():
            return folder / name
    raise FileNotFoundError(f"{name} not found. Put it into a folder 'data' next to this notebook.")


def plot_stack(weights, title):
    """How often each weight was used, drawn like the worn paint on a weight stack."""
    shown = weights[weights.isin(STACK)]
    counts = shown.value_counts().reindex(STACK, fill_value=0)
    fig, ax = plt.subplots(figsize=(5, 6))
    ax.barh(counts.index.astype(str), counts.values, left=-counts.values / 2, color="#b08d57")
    ax.invert_yaxis()
    ax.set_xticks([])
    ax.set_ylabel("weight (kg)")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def make_gym_log(seed=7):
    """Exercises at four machines of a gym. One row = one member at one machine on one day:
    the weight, the number of sets (series) and the repetitions per set (made-up data)."""
    rng = np.random.default_rng(seed)

    def common(machine, n, mean_w, sd_w, low, high, step, reps_at_mean=10, reps_per_kg=0.08):
        w = np.clip(np.round(rng.normal(mean_w, sd_w, n) / step) * step, low, high)
        reps = np.clip(np.round(reps_at_mean - reps_per_kg * (w - mean_w) + rng.normal(0, 1.5, n)), 5, 15)
        sets = rng.choice([2, 3, 4, 5], n, p=[0.15, 0.45, 0.3, 0.1])
        return pd.DataFrame({"machine": machine, "weight_kg": w, "sets": sets, "reps": reps})

    chest = common("Chest press", 820, 55, 12, 15, 100, 5)
    # real but rare: very strong (max strength: heavy, few reps), very light (rehab: light, many reps),
    # endurance (medium weight, many reps), bodybuilders (many sets)
    strong = pd.DataFrame({"machine": "Chest press", "weight_kg": rng.choice([110, 115, 120, 125, 130], 30),
                           "sets": rng.choice([3, 4, 5], 30), "reps": rng.integers(1, 4, 30)})
    light = pd.DataFrame({"machine": "Chest press", "weight_kg": rng.choice([5, 10], 20),
                          "sets": rng.choice([2, 3], 20), "reps": rng.integers(25, 51, 20)})
    endurance = pd.DataFrame({"machine": "Chest press", "weight_kg": rng.choice([30, 35, 40], 25),
                              "sets": rng.choice([2, 3], 25), "reps": rng.integers(20, 31, 25)})
    bodybuilders = common("Chest press", 15, 60, 8, 40, 90, 5)
    bodybuilders["sets"] = rng.integers(6, 11, 15)
    # errors and a strange value
    odd = pd.DataFrame({"machine": "Chest press",
                        "weight_kg": [1300, 1300, 1300, 0, 0, 0, 0, 57, 57, 50, 60, 45, 55, 50, 65],
                        "sets": [3, 3, 4, 3, 3, 2, 3, 3, 3, 3, 3, 3, 3, 30, 3],
                        "reps": [10, 8, 10, 10, 12, 10, 9, 10, 10, 0, 0, 0, 120, 10, 120]})
    parts = [chest, strong, light, endurance, bodybuilders, odd,
             common("Lat pulldown", 700, 50, 12, 15, 100, 5),
             common("Biceps curl", 500, 20, 6, 5, 40, 5, reps_per_kg=0.2),
             common("Leg press", 600, 160, 35, 60, 300, 10, reps_per_kg=0.03)]
    gym = pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)
    gym[["weight_kg", "sets", "reps"]] = gym[["weight_kg", "sets", "reps"]].astype(int)
    gym.insert(0, "member_id", rng.integers(100, 400, len(gym)))
    return gym


def make_bookings(seed=4):
    """A new export of the Kiez Gym booking system (made-up data)."""
    rng = np.random.default_rng(seed)
    n = 1500
    # members: plan and months as member stay the same for one person
    members = pd.DataFrame({"member_id": np.arange(100, 500)})
    members["plan"] = rng.choice(["Basic", "Plus", "Premium"], len(members), p=[0.5, 0.3, 0.2])
    members["months_member"] = np.clip(rng.gamma(2.0, 9, len(members)).round(), 1, 72).astype(int)
    members["habit"] = rng.normal(0, 0.6, len(members))
    members["past_no_shows"] = rng.poisson(np.exp(members["habit"]) * 1.2)
    extra = np.random.default_rng(seed + 200)
    founders = extra.choice(members.index, 15, replace=False)            # real: members since the gym opened
    members.loc[founders, "months_member"] = extra.integers(120, 181, 15)
    rest = np.setdiff1d(members.index, founders)
    chronic = extra.choice(rest, 14, replace=False)                      # real: members who often do not come
    members.loc[chronic, "past_no_shows"] = extra.integers(8, 21, 14)
    members.loc[chronic, "habit"] = members.loc[chronic, "habit"] + 0.8

    df = pd.DataFrame({"member_id": rng.choice(members["member_id"], n)})
    df = df.merge(members, on="member_id", how="left")
    days = pd.date_range("2026-04-06", "2026-06-28")
    df["date"] = rng.choice(days, n)
    df["class_type"] = rng.choice(["Spinning", "Yoga", "HIIT", "Pilates"], n, p=[0.35, 0.25, 0.25, 0.15])
    df["class_time"] = rng.choice(["07:00", "09:00", "12:30", "18:00", "19:30"], n, p=[0.2, 0.15, 0.1, 0.3, 0.25])
    _ = np.where(rng.random(n) < 0.5, rng.uniform(0.5, 24, n), rng.uniform(24, 96, n))   # keeps the random stream
    # many people book shortly before the class, few book days before: an exponential distribution
    df["hours"] = np.round(np.clip(np.random.default_rng(seed + 100).exponential(24, n), 0.5, 168), 1)

    # who does not come
    weekday = df["date"].dt.dayofweek
    z = (-2.93 + 0.7 * (df["plan"] == "Basic") - 0.6 * (df["plan"] == "Premium")
         + 0.8 * (df["class_time"] == "07:00")
         + 0.9 * ((weekday == 4) & (df["class_time"].isin(["18:00", "19:30"])))
         + 0.9 * (df["hours"] > 24) + 0.8 * (df["months_member"] <= 3)
         + 0.35 * np.minimum(df["past_no_shows"], 6) + 0.6 * df["habit"])
    df["no_show"] = (rng.random(n) < 1 / (1 + np.exp(-z))).astype(int)

    # check in and check out, only for people who came
    start = pd.to_datetime(df["date"].dt.strftime("%Y-%m-%d") + " " + df["class_time"])
    cin = start + pd.to_timedelta(rng.integers(-10, 6, n), unit="m")
    cout = cin + pd.to_timedelta(rng.integers(50, 80, n), unit="m")
    came = df.index[df["no_show"] == 0]
    forgot = rng.choice(came, 6, replace=False)                    # forgot to scan out: system closes at 23:59
    cout[forgot] = pd.to_datetime(df.loc[forgot, "date"].dt.strftime("%Y-%m-%d") + " 23:59")
    rest = np.setdiff1d(came, forgot)
    typo = rng.choice(rest, 3, replace=False)                      # typing error: out before in
    cout[typo] = cin[typo] - pd.to_timedelta(60, unit="m")
    fmt = "%Y-%m-%d %H:%M"
    df["check_in"] = np.where(df["no_show"] == 0, cin.dt.strftime(fmt), None)
    df["check_out"] = np.where(df["no_show"] == 0, cout.dt.strftime(fmt), None)

    # the messy table, as it comes out of the booking system
    out = pd.DataFrame({
        "booking_id": [f"B{10001 + i}" for i in range(n)],
        "member_id": df["member_id"],
        "plan": df["plan"].astype(object),
        "months_member": df["months_member"].astype(str) + " months",
        "class_type": df["class_type"].astype(object),
        "class_date": df["date"].dt.strftime("%Y-%m-%d"),
        "class_time": df["class_time"],
        "hours_booked_before": df["hours"].astype(str) + " h",
        "past_no_shows": df["past_no_shows"],
        "check_in": df["check_in"],
        "check_out": df["check_out"],
        "no_show": df["no_show"],
    })
    idx = rng.permutation(n)
    out.loc[idx[:80], "plan"] = "Unknown"
    out.loc[idx[80:110], "months_member"] = np.nan
    out.loc[idx[110:140], "class_type"] = out.loc[idx[110:140], "class_type"].str.lower() + " "
    out.loc[idx[140:160], "class_type"] = "SPIN"
    out.loc[idx[160:185], "plan"] = out.loc[idx[160:185], "plan"].str.upper()
    out.loc[idx[185:205], "plan"] = " " + out.loc[idx[185:205], "plan"].str.lower()
    out.loc[idx[205], "past_no_shows"] = 99
    true_h = df["hours"].values
    cand = [i for i in idx[210:] if 3 <= true_h[i] <= 48][:60]          # error: saved in minutes, not hours
    out.loc[cand, "hours_booked_before"] = (np.round(true_h[cand] * 60, 1)).astype(str) + " h"
    code999 = idx[400:440]                                              # error: 999 = "we do not know"
    out.loc[code999, "months_member"] = "999 months"
    dup = out.sample(40, random_state=1)
    out = pd.concat([out, dup]).sort_values("booking_id").reset_index(drop=True)
    return out


def look_ahead(tables, top=100):
    """tables = {"name": table, ...}. Train the same two models on every table.
    Then take the top bookings with the highest risk in the test part and count the real no-shows."""
    from sklearn.model_selection import train_test_split
    from sklearn.linear_model import LogisticRegression
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import make_pipeline

    def features(t):
        X = pd.DataFrame({
            "plan": t["plan"].map({"basic": 0, "plus": 1, "premium": 2, "unknown": 1}),
            "months_member": t["months_member"],
            "hours_booked_before": t["log_hours"] if "log_hours" in t else t["hours_booked_before"],
            "past_no_shows": t["past_no_shows"],
            "friday": (pd.to_datetime(t["class_date"]).dt.dayofweek == 4).astype(int),
            "early_class": (t["class_time"] == "07:00").astype(int),
        })
        return X.fillna(X.median())

    rows = {}
    for name, t in tables.items():
        X, y = features(t), t["no_show"]
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=4, stratify=y)
        result = {}
        for model_name, model in [("Logistic regression", make_pipeline(StandardScaler(), LogisticRegression())),
                                  ("Decision tree", DecisionTreeClassifier(max_depth=4, random_state=0))]:
            model.fit(X_train, y_train)
            risk = model.predict_proba(X_test)[:, 1]
            best = np.argsort(-risk)[:top]
            result[model_name] = f"{int(y_test.iloc[best].sum())} of {top}"
        rows[name] = result
    table = pd.DataFrame(rows).T
    print(f"Real no-shows among the {top} bookings with the highest risk (test data, never seen by the model):")
    return table


def model_test(data, target, inputs, top=100):
    """Train a logistic regression on the input columns (all scaled to 0 ... 1).
    Return how many real positives (no-shows, defaults) are among the top rows with the highest risk in test data."""
    from sklearn.model_selection import train_test_split
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import MinMaxScaler
    from sklearn.pipeline import make_pipeline
    X = data[inputs].fillna(data[inputs].median())
    y = data[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=4, stratify=y)
    model = make_pipeline(MinMaxScaler(), LogisticRegression(max_iter=1000)).fit(X_train, y_train)
    risk = model.predict_proba(X_test)[:, 1]
    best = np.argsort(-risk)[:top]
    return int(y_test.iloc[best].sum())


gym = make_gym_log()
print("Ready:", len(gym), "sets at", gym["machine"].nunique(), "machines")

---
# Part 1 · One machine, many people (introduction, just run)

Maybe you have seen the photo of a gym weight stack: the paint is worn off in the middle and almost new at the top and the bottom. Most people use the weights in the middle. A few use the very light or the very heavy plates.

**One row = one member at one machine on one day:**

| Column | Meaning |
|---|---|
| `weight_kg` | the weight on the machine |
| `sets` | how many series the member did |
| `reps` | how many repetitions in each series |

Today we look at the **chest press**. First we find and treat the **errors**. Then we study the **rare but real** values, and what they do to our numbers and predictions.

**Example**

In [ ]:
# Step 1: the first rows
gym.head()

In [ ]:
# Step 2: only the chest press. .copy() makes a real new table we can change
chest = gym[gym["machine"] == "Chest press"].copy()
print(len(chest), "rows at the chest press")

In [ ]:
# Step 3: how often each plate of the stack was used, drawn like the worn paint
plot_stack(chest["weight_kg"], "Chest press: how often each weight was used")

Wide in the middle (around 55 kg), thin at both ends. The plates at the bottom (110 to 130 kg) and at the top (5 and 10 kg) are used, but rarely.

## 1.1 · First the errors: detect, judge, treat
We use **domain rules**: what we know about the gym.

| Column | Rule |
|---|---|
| `weight_kg` | must be a plate of the stack: 5, 10, ..., 130 kg |
| `sets` | 1 to 15 |
| `reps` | 1 to 60 |

A rule only **detects**. Then **we judge** each case and choose the treatment.

**Example**

In [ ]:
# Step 1: weights that are not on the stack. ~ means "not", .isin(STACK) is True if the weight is in the list
flag_weight = ~chest["weight_kg"].isin(STACK)
print("weights not on the stack:", flag_weight.sum())
chest[flag_weight]["weight_kg"].value_counts()

In [ ]:
# Step 2: impossible repetitions. | means "or"
flag_reps = (chest["reps"] < 1) | (chest["reps"] > 60)
chest[flag_reps]

In [ ]:
# Step 3: impossible sets
flag_sets = (chest["sets"] < 1) | (chest["sets"] > 15)
chest[flag_sets]

Now we **judge** every case:

| Value | Our judgment | Treatment |
|---|---|---|
| 1300 kg (3 rows) | typo: 130 kg with one zero too many | **repair** to 130 |
| 0 kg (4 rows) | impossible, the true weight is unknown | **empty** |
| 57 kg (2 rows) | not on the stack, but the machine has a small add-on plate of 2.5 kg: 55 + 2.5 = 57.5 | **keep**: possible |
| 0 reps (3 rows) | impossible: a set has at least one repetition | **empty** |
| 120 reps (2 rows) | typo: 12 with one zero too many (with 55 and 65 kg nobody does 120) | **repair** to 12 |
| 30 sets (1 row) | typo: 3 with one zero too many | **repair** to 3 |

The rule flagged the 57 kg, but the rule is not the judge. **You** are.

`.replace(old, new)` changes every old value into the new one. `np.nan` means "empty, no value".

**Example**

In [ ]:
# Step 4: treat the weight
chest["weight_kg"] = chest["weight_kg"].replace(1300, 130)
chest["weight_kg"] = chest["weight_kg"].replace(0, np.nan)
print("empty weights:", chest["weight_kg"].isna().sum())

In [ ]:
# Step 5: treat the repetitions and the sets
chest["reps"] = chest["reps"].replace(120, 12)
chest["reps"] = chest["reps"].replace(0, np.nan)
chest["sets"] = chest["sets"].replace(30, 3)
print("empty reps:", chest["reps"].isna().sum())

In [ ]:
# Step 6: check: the three columns after the treatment
chest[["weight_kg", "sets", "reps"]].describe().round(1)

No more impossible values: weight 5 to 130 kg, sets 2 to 10, reps 1 to 47. From now on we work with this repaired table. What is still extreme is **real**.

## 1.2 · See the distributions
A boxplot shows a column in a short form. First the simple picture.

**Example**

In [ ]:
# Step 1: a simple boxplot of the weight
chest["weight_kg"].plot(kind="box", vert=False, figsize=(7, 2), title="Chest press: weight")
plt.show()

A box, a line inside it, two whiskers and some dots. What does each part mean? To name the parts we need the **quartiles**: `.quantile(0.25)` gives the value with 25% of the rows below it, `.quantile(0.75)` the value with 75% below it.

**Example**

In [ ]:
# Step 2: the same boxplot, with every part named
data = chest["weight_kg"].dropna()       # dropna() leaves out the empty values
q1 = data.quantile(0.25)
q2 = data.median()
q3 = data.quantile(0.75)
lower_fence = q1 - 1.5 * (q3 - q1)
upper_fence = q3 + 1.5 * (q3 - q1)
whisker_low = data[data >= lower_fence].min()     # smallest value inside the fence
whisker_high = data[data <= upper_fence].max()    # biggest value inside the fence

data.plot(kind="box", vert=False, figsize=(10, 3.5), title="Chest press: weight")
above = {"min": data.min(), "whisker": whisker_low, "whisker ": whisker_high, "max": data.max()}
below = {"Q1": q1, "Q2 = median": q2, "Q3": q3}
for name, value in above.items():
    plt.annotate(f"{name}\n{value:g} kg", xy=(value, 1.08), xytext=(value, 1.35),
                 ha="center", fontsize=9, arrowprops=dict(arrowstyle="->"))
for i, (name, value) in enumerate(below.items()):
    plt.annotate(f"{name}\n{value:g} kg", xy=(value, 0.92), xytext=(value + (i - 1) * 6, 0.6),
                 ha="center", fontsize=9, arrowprops=dict(arrowstyle="->"))
plt.axvline(lower_fence, color="red", linestyle="--")
plt.axvline(upper_fence, color="red", linestyle="--")
plt.text(lower_fence, 1.5, f"cut {lower_fence:g} kg", color="red", ha="center")
plt.text(upper_fence, 1.5, f"cut {upper_fence:g} kg", color="red", ha="center")
plt.ylim(0.45, 1.6)
plt.yticks([])
plt.show()

**How to read a boxplot:**
- **Q1**: 25% of the rows are lower. **Q2 = median**: half are lower. **Q3**: 75% are lower.
- The **box** holds the middle 50%, from Q1 to Q3. Its length (Q3 - Q1) is the **IQR**.
- The red lines are the **cuts**: 1.5 box lengths below Q1 and above Q3.
- The **whiskers** go to the last real value inside the cuts, not to the cut itself.
- Every **dot** outside the cuts is a possible outlier. There is no "Q4": the end of the data is the **max**.

**Example**

In [ ]:
# Step 3: the repetitions: histogram and boxplot
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
chest["reps"].plot(kind="hist", bins=40, ax=ax[0], title="Repetitions per set")
chest["reps"].plot(kind="box", vert=False, ax=ax[1], title="Repetitions per set")
plt.show()

In [ ]:
# Step 4: the sets: how many rows for each number of sets
chest["sets"].value_counts().sort_index().plot(kind="bar", rot=0, figsize=(7, 3), title="Sets per exercise")
plt.xlabel("sets")
plt.show()

- **Repetitions:** most members do 8 to 12. Some do **25 to 50** (endurance training, rehabilitation with light weights). A few do only **1 to 3** (testing their maximum strength with heavy weights).
- **Sets:** most do 3 or 4. A few do **6 to 10** (bodybuilders).

None of these are errors. They are different ways to train.

## 1.3 · Rare but real: what the IQR finds
The IQR rule from the boxplot, step by step, for the repetitions.

**Example**

In [ ]:
# Step 1: the quartiles
q1 = chest["reps"].quantile(0.25)
q3 = chest["reps"].quantile(0.75)
print("Q1:", q1, "  Q3:", q3)

In [ ]:
# Step 2: the box length (IQR) and the two cuts
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
print("IQR:", iqr, "  lower cut:", lower, "  upper cut:", upper)

In [ ]:
# Step 3: flag every row outside the cuts and count
flag_reps_outlier = (chest["reps"] < lower) | (chest["reps"] > upper)
print("IQR finds:", flag_reps_outlier.sum(), "rows")

In [ ]:
# Step 4: who are they? Their weight and their repetitions
chest[flag_reps_outlier].groupby("reps")["weight_kg"].mean().round(0)

Step 4 shows the mean weight for each number of repetitions among the flagged rows:
- **1 to 3 reps** with about **120 kg**: maximum strength;
- **20 to 29 reps** with about **25 to 40 kg**: endurance;
- **30 to 47 reps** with **5 to 10 kg**: rehabilitation.

The IQR finds them all, and all of them are real. **The IQR tells you where to look. It does not tell you what is wrong.**

**The z-score** is another detection method: how many standard deviations a value is away from the mean. A z bigger than 3 or smaller than -3 counts as an outlier.

z = (value - mean) / standard deviation

**Example**

In [ ]:
# Step 5: the z-score of every row, as a new column at the end of the table
# chest["reps"].mean()        -> the mean of the repetitions
# chest["reps"].std()         -> the standard deviation: the typical distance from the mean
# (reps - mean) / std         -> how many standard deviations each row is away from the mean
# .round(1)                   -> one decimal
chest["z_reps"] = ((chest["reps"] - chest["reps"].mean()) / chest["reps"].std()).round(1)
chest.head(10)

In [ ]:
# Step 6: flag every row with z bigger than 3 or smaller than -3, and count
flag_reps_z = chest["z_reps"].abs() > 3            # abs() removes the minus sign: -4 becomes 4
print("z-score finds:", flag_reps_z.sum(), "rows   (IQR found", flag_reps_outlier.sum(), ")")

In [ ]:
# Step 7: look at the rows the z-score found, from the most extreme down
# chest[flag_reps_z]                              -> only the rows with |z| bigger than 3
# .sort_values("z_reps", ascending=False)         -> the biggest z first
chest[flag_reps_z].sort_values("z_reps", ascending=False)

The z-score finds fewer: only the most extreme high values (Step 7 starts at the top and never reaches anybody with 1 to 3 reps). It never flags them, because the long tail to the right makes the standard deviation big. Different methods, different answers: one more reason to look at the rows yourself.

## 1.4 · The impact of real outliers
They are real. So why care? Because they change the numbers we report and the predictions we make. Three impacts, each one **picture first**, numbers after.

**Impact 1: the average.** Where are the mean and the median of the repetitions?

**Example**

In [ ]:
# Step 1: picture: histogram of the repetitions, a red line for the mean, a black one for the median
chest["reps"].plot(kind="hist", bins=45, figsize=(8, 3), title="Repetitions per set")
plt.axvline(chest["reps"].mean(), color="red", label="mean")
plt.axvline(chest["reps"].median(), color="black", label="median")
plt.legend()
plt.show()

In [ ]:
# Step 2: the numbers, with and without the IQR outliers
compare = pd.DataFrame({
    "all rows": [chest["reps"].mean(), chest["reps"].median()],
    "without the outliers": [chest[~flag_reps_outlier]["reps"].mean(), chest[~flag_reps_outlier]["reps"].median()],
}, index=["mean", "median"]).round(1)
compare

The red line sits to the right of the black one: a few members with 20 to 47 reps pull the mean. Without them the mean falls from 10.8 to 10.1; the median stays at 10. If the app says *"our members do on average 10.8 repetitions"*, the typical member does 10.

**Impact 2: a prediction.** The app wants to suggest how many repetitions a member can do with a given weight. The simplest model is a **straight line** through the points.

**Example**

In [ ]:
# Step 3: picture: weight against repetitions. Red = IQR outliers
data = chest.dropna(subset=["weight_kg", "reps"])     # only rows with weight and reps
outlier = (data["reps"] < lower) | (data["reps"] > upper)
plt.figure(figsize=(8, 4))
plt.scatter(data[~outlier]["weight_kg"], data[~outlier]["reps"], alpha=0.3, label="common")
plt.scatter(data[outlier]["weight_kg"], data[outlier]["reps"], color="red", label="IQR outliers")
plt.xlabel("weight (kg)"); plt.ylabel("repetitions"); plt.legend()
plt.show()

In [ ]:
# Step 4: picture: two straight lines, one through ALL points, one WITHOUT the outliers
# np.polyfit(x, y, 1) finds the best straight line, np.polyval(line, x) reads the line at x
line_all = np.polyfit(data["weight_kg"], data["reps"], 1)
line_common = np.polyfit(data[~outlier]["weight_kg"], data[~outlier]["reps"], 1)
x = np.arange(5, 135)
plt.figure(figsize=(8, 4))
plt.scatter(data[~outlier]["weight_kg"], data[~outlier]["reps"], alpha=0.3, label="common")
plt.scatter(data[outlier]["weight_kg"], data[outlier]["reps"], color="red", label="IQR outliers")
plt.plot(x, np.polyval(line_all, x), color="red", label="line with all points")
plt.plot(x, np.polyval(line_common, x), color="black", label="line without outliers")
plt.xlabel("weight (kg)"); plt.ylabel("repetitions"); plt.legend()
plt.show()

In [ ]:
# Step 5: the numbers: what does each line predict for 30 kg and for 80 kg?
for weight in [30, 80]:
    print(weight, "kg  ->  with all:", round(np.polyval(line_all, weight), 1),
          "reps   without outliers:", round(np.polyval(line_common, weight), 1), "reps")

The red line is much steeper: the rehabilitation members (5 to 10 kg, 40 reps) pull its left end up, the strength testers (120 kg, 2 reps) pull its right end down. At 130 kg it even goes **below zero**, an impossible prediction.

For 80 kg: 6.3 reps against 8.3. For 30 kg: 15.2 against 11.8. Less than 10% of the rows (75 of 918) change what the app tells **every** member.

**Impact 3: the scale.** Later in the semester we **scale** every column to values between 0 and 1, so a model can compare columns with different units (kg and repetitions):

scaled = (value - min) / (max - min)

Here the min and the max are both outliers (1 rep and 47 reps).

**Example**

In [ ]:
# Step 6: picture: the scaled repetitions, all values kept and after a cap at the IQR cuts
reps = data["reps"]
scaled = (reps - reps.min()) / (reps.max() - reps.min())
capped_reps = reps.clip(lower=lower, upper=upper)      # cap: explained step by step in 1.5
scaled_capped = (capped_reps - capped_reps.min()) / (capped_reps.max() - capped_reps.min())

fig, ax = plt.subplots(1, 2, figsize=(11, 3), sharey=True)
scaled.plot(kind="hist", bins=30, ax=ax[0], title="scaled, all values kept")
scaled_capped.plot(kind="hist", bins=30, ax=ax[1], title="scaled, after the cap")
ax[0].set_xlim(0, 1); ax[1].set_xlim(0, 1)
plt.show()

In [ ]:
# Step 7: the numbers: where do 8, 10 and 12 reps land?
for value in [8, 10, 12]:
    print(value, "reps  ->  all kept:", round((value - reps.min()) / (reps.max() - reps.min()), 2),
          "   after the cap:", round((value - capped_reps.min()) / (capped_reps.max() - capped_reps.min()), 2))

Left: almost the whole gym is squeezed between 0.1 and 0.3, because one member did 47 reps. 8, 10 and 12 reps become **0.15, 0.20 and 0.24**: for a model, the difference almost disappears. Right, after the cap: **0.29, 0.46 and 0.62**, the members use the whole scale again.

So real outliers distort **three** things: the **mean** (not the median), the **scale**, and the **predictions** built on them. The cap fixed the scale. How does it work exactly?

## 1.5 · Capping step by step
**Capping** (also called *winsorizing*) means: every value below a lower limit becomes the lower limit, every value above an upper limit becomes the upper limit. The row stays, only the extreme value is pulled back to the limit.

The usual limits are the **IQR cuts** from the boxplot. For the repetitions we already have them (1.3): `lower` and `upper`.

**Example**

In [ ]:
# Step 1: the caps
print("lower cap:", lower, "reps")
print("upper cap:", upper, "reps")

In [ ]:
# Step 2: which rows will change? Everything outside the caps
# Which rows will the cap change? Only the rows **outside** the caps: fewer than 4.5 reps or more than 16.5 reps. We find them in three small steps.

In [ ]:
# Step 2a: True/False for every row. True = outside the caps
# chest["reps"] < lower    -> True if the row has fewer reps than the lower cap
# chest["reps"] > upper    -> True if the row has more reps than the upper cap
# |                        -> "or": True if one of the two is True
to_change = (chest["reps"] < lower) | (chest["reps"] > upper)
to_change.head()

`to_change` is one column with `True` or `False` for every row. The first rows are all `False`: normal sets.

In [ ]:
# Step 2b: count the rows that will change. .sum() counts the Trues (True = 1, False = 0)
print(to_change.sum(), "rows will change")

In [ ]:
# Step 2c: look at these rows, with all columns
# chest[to_change]       -> keep only the rows where to_change is True
# .sort_values("reps")   -> sort from the fewest to the most reps
# .head(8)               -> show the first 8 rows
chest[to_change].sort_values("reps").head(8)

75 rows will change. The first ones are the strength testers: 110 to 130 kg with only 1 rep. At the end of the list (try `.tail(8)`) are the rehabilitation members: 5 to 10 kg with more than 40 reps.

In [ ]:
# Step 3a: replace the values outside the caps, in a NEW column
# chest["reps"]                        -> the original repetitions
# .clip(lower=lower, upper=upper)      -> every value below 4.5 becomes 4.5,
#                                         every value above 16.5 becomes 16.5,
#                                         all other values stay as they are
# chest["reps_capped"] = ...           -> save the result in a new column; the original stays
chest["reps_capped"] = chest["reps"].clip(lower=lower, upper=upper)
chest.head()

In [ ]:
# Step 3b: the normal rows did not change
# The first rows are normal sets: reps and reps_capped are the same.
# Check it for the whole table: how many rows are different now?
# !=      -> "not equal": True if the two columns are different in this row
# .sum()  -> counts the Trues
print((chest["reps"] != chest["reps_capped"]).sum(), "rows changed")

In [ ]:
# Step 3c: look only at the rows that changed, with all columns
# chest[to_change]     -> only the 75 rows outside the caps (from Step 2a)
# .sort_values("reps") -> from the fewest to the most reps
# The new column reps_capped is at the end, on the right:
# 1, 2 and 3 reps became 4.5; everything from 20 to 47 reps became 16.5
chest[to_change].sort_values("reps")

Look at the table: 1, 2 and 3 reps became **4.5**, everything from 20 to 47 became **16.5**. All other rows did not change. We wrote the result in a **new column**, so the original stays and we can compare.

**Example**

In [ ]:
# Step 4: picture: boxplot before and after the cap
chest[["reps", "reps_capped"]].plot(kind="box", vert=False, figsize=(9, 3), title="Repetitions: before and after the cap")
plt.show()

In [ ]:
# Step 5: the numbers before and after
chest[["reps", "reps_capped"]].describe().round(1)

- **Boxplot:** the box and the median are the same. The dots are gone: the whiskers now end exactly at the caps.
- **Numbers:** count, median, Q1 and Q3 do not change. **min, max, mean and std** do. The standard deviation falls from 5.0 to 2.4: the column is no longer stretched by a few people.

| Cap | Strength | Weakness |
|---|---|---|
| keeps every row | no member is deleted | a 47-rep member now looks like a 16.5-rep member |
| uses the IQR cuts | the same limits as the boxplot | other limits are possible, for example the 1% and 99% quantiles: `.quantile(0.01)` and `.quantile(0.99)` |

## 1.6 · Capping with many categories
Now the **weight** of **all four machines**. A leg press moves 160 kg, a biceps curl 20 kg. First we repair the errors of the whole table, as in 1.1.

**Example**

In [ ]:
# Step 1: a copy of the whole gym with the weight errors repaired
gym_fixed = gym.copy()
gym_fixed["weight_kg"] = gym_fixed["weight_kg"].replace(1300, 130).replace(0, np.nan)
gym_fixed.groupby("machine")["weight_kg"].describe().round(0)

In [ ]:
# Step 2: picture: one boxplot per machine. by="machine" draws one box for each machine
gym_fixed.boxplot(column="weight_kg", by="machine", vert=False, figsize=(8, 3.5))
plt.suptitle("")
plt.title("Weight per machine")
plt.show()

**The trap:** one cap for all machines together.

**Example**

In [ ]:
# Step 3: one upper cap, calculated with ALL machines
w = gym_fixed["weight_kg"]
upper_all = w.quantile(0.75) + 1.5 * (w.quantile(0.75) - w.quantile(0.25))
print("one upper cap for all machines:", upper_all, "kg")

In [ ]:
# Step 4: how many rows would this cap change, per machine?
gym_fixed[gym_fixed["weight_kg"] > upper_all]["machine"].value_counts()

This cap would turn **446 normal leg press sets** into 135 kg. Totally wrong: the leg press is simply heavier. Every machine needs **its own caps**.

**Example**

In [ ]:
# Step 5: Q1 and Q3 for every machine
# groupby("machine") splits the table by machine, .quantile([0.25, 0.75]) works inside each machine,
# .unstack() puts Q1 and Q3 next to each other as columns
caps = gym_fixed.groupby("machine")["weight_kg"].quantile([0.25, 0.75]).unstack()
caps.columns = ["q1", "q3"]
caps

In [ ]:
# Step 6: the lower and the upper cap for every machine
caps["lower"] = caps["q1"] - 1.5 * (caps["q3"] - caps["q1"])
caps["upper"] = caps["q3"] + 1.5 * (caps["q3"] - caps["q1"])
caps

This table **is** the answer to *"what are the caps?"*: 22.5 to 82.5 kg for the chest press, 55 to 255 kg for the leg press, and so on. Now we apply each machine's caps to the rows of that machine.

`for machine in caps.index:` repeats the indented lines once for each machine. `.loc[rows, "column"]` changes only the rows of that machine.

**Example**

In [ ]:
# Step 7: cap every machine with its own limits, in a new column
# gym_fixed["weight_capped"] = gym_fixed["weight_kg"]  -> start with a copy of the weights
# for machine in caps.index:                           -> repeat for every machine in the caps table
#     rows = gym_fixed["machine"] == machine           -> True for the rows of this machine
#     .loc[rows, "weight_capped"] = ... .clip(...)     -> cap only these rows, with this machine's lower and upper cap
gym_fixed["weight_capped"] = gym_fixed["weight_kg"]
for machine in caps.index:
    rows = gym_fixed["machine"] == machine
    gym_fixed.loc[rows, "weight_capped"] = gym_fixed.loc[rows, "weight_kg"].clip(
        lower=caps.loc[machine, "lower"], upper=caps.loc[machine, "upper"])
gym_fixed.head()

In [ ]:
# Step 8: how many rows changed, per machine?
gym_fixed["changed"] = gym_fixed["weight_kg"] != gym_fixed["weight_capped"]
gym_fixed[gym_fixed["weight_kg"].notna()].groupby("machine")["changed"].sum()

In [ ]:
# Step 9: picture: boxplots per machine, before and after the cap
fig, ax = plt.subplots(1, 2, figsize=(13, 3.5), sharey=True)
gym_fixed.boxplot(column="weight_kg", by="machine", vert=False, ax=ax[0])
gym_fixed.boxplot(column="weight_capped", by="machine", vert=False, ax=ax[1])
ax[0].set_title("before the cap"); ax[1].set_title("after the cap, each machine with its own caps")
plt.suptitle("")
plt.show()

Only **60 rows** changed: 58 chest press sets (the very light and the very strong members) and 2 leg press sets (260 and 290 kg). Biceps curl and lat pulldown had nothing outside their caps. In the picture, the dots are gone, and every machine keeps its own range: the leg press still goes up to 255 kg.

With one cap for all, 446 rows would have changed, and all of them wrongly.

## 1.7 · Keep, cap or remove? Measure the effect
A rare value is **not** a reason to delete. Think of salaries: the CEO earns 20 times more than everybody else. Keep it or not? Ask four questions, in this order:

| Question | If the answer is ... | Example |
|---|---|---|
| 1. Is it **possible**? | no: it is an error, repair or replace it | 1300 kg, 0 kg, 0 reps |
| 2. Is it part of the **group you study**? | no: leave it out | the CEO in a model for analyst salaries |
| 3. Does it carry the **information you want**? | yes: keep it | endurance members, if the app also coaches them |
| 4. Does it **break your method**? | yes: cap it or transform it | the mean, a straight line, scaling |

Here are three treatments for the repetitions, and what each one does to the mean and to the prediction for 80 kg.

**Example**

In [ ]:
# Step 1: three versions of the data
keep = data
capped = data.assign(reps=data["reps"].clip(lower=lower, upper=upper))   # cap at the IQR cuts
removed = data[~outlier]                                                   # remove the outlier rows
print("rows:  keep", len(keep), "  cap", len(capped), "  remove", len(removed))

In [ ]:
# Step 2: the mean and the prediction for 80 kg, for each version
results = pd.DataFrame({
    "rows": [len(keep), len(capped), len(removed)],
    "mean reps": [keep["reps"].mean(), capped["reps"].mean(), removed["reps"].mean()],
    "reps at 80 kg": [np.polyval(np.polyfit(t["weight_kg"], t["reps"], 1), 80) for t in [keep, capped, removed]],
}, index=["keep", "cap", "remove"]).round(1)
results

| Technique | Strength | Weakness |
|---|---|---|
| **keep** | honest: every member counts | the mean and the line are pulled by a few members |
| **cap** (`clip`) | keeps every row, removes the pull | changes real values: a 40-rep member becomes a 16.5-rep member |
| **remove** | the line fits the typical member | the app knows nothing about endurance and rehab members |
| **repair** | the true value comes back | only for errors with a known cause (1300 → 130) |
| **replace with the median** | keeps the row | only for errors, hides the variation |
| **log** | keeps every value, shortens a long tail | harder to read |

There is no single right answer. It depends on the question: *"How many reps for a typical member?"* → cap or remove. *"How do all our members train?"* → keep. Often the best solution is a **new column** that says which kind of member it is (strength, endurance, rehab), so a model can treat them differently.

In Part 2 you use these techniques yourself.

---
# Part 2 · Kiez Gym: measure, decide, replace

Back to the no-show case from last week. The gym sent a **new export** of its booking system, over a longer period. First we bring the table to the same state as at the end of Tutorial 02.

**Example**

In [ ]:
# Step 1: the new export
bookings = make_bookings()
bookings.shape

In [ ]:
# Step 2: remove the duplicates using drop_duplicates(). .copy() makes a real new table we can change
bookings = bookings._____________().copy()
bookings.shape

In [ ]:
# Step 3: clean the text of plan and class type (spaces, capital letters, "spin")
# .str.strip()              -> removes spaces at the start and at the end
# .str.lower()              -> writes everything in small letters
# .replace("spin", "spinning") -> "spin" is a short name for "spinning"
bookings["plan"] = bookings["plan"].str.strip().str.lower()
bookings["class_type"] = bookings["class_type"].str.strip().str.lower().replace("spin", "spinning")
bookings.head()

In [ ]:
# Step 4: numbers saved as text become numbers
# .str.replace(" months", "")  -> remove the text " months", only the number is left (still as text)
# pd.to_numeric(...)           -> turn the text into a real number
bookings["months_member"] = pd.to_numeric(bookings["months_member"].str.replace(" months", ""))
bookings["hours_booked_before"] = pd.to_numeric(bookings["hours_booked_before"].str.replace(" h", ""))
bookings.head()

In [ ]:
# Step 5: the weekday and the flag for the unknown plans, as last week
bookings["weekday"] = pd.to_datetime(bookings["class_date"]).dt.day_name()
bookings["flag_plan_unknown"] = bookings["plan"] == "unknown"
bookings[["past_no_shows", "months_member", "hours_booked_before"]].describe().round(1)

Look at the **max** row: **99** earlier no-shows, **999** months as member (83 years!), **2,856** hours booked before (four months; the app allows one week). These are **errors**, and finding errors is what you did last week. So we treat them in one cell, like a recipe:

| Error | Judgment | Treatment |
|---|---|---|
| 99 earlier no-shows | system error, true value unknown | empty, then the median |
| 999 months as member | code for "we do not know" | empty, then the median |
| more than 168 hours booked before | saved in **minutes** instead of hours | **repair**: divide by 60 |

**Example**

In [ ]:
# Step 6: keep a copy of the export as it is. We compare with it in Part 3
raw = bookings.copy()

In [ ]:
# Step 7: treat the errors. .replace(old, new), np.nan = empty, .fillna(x) puts x into every empty cell
bookings["past_no_shows"] = bookings["past_no_shows"].replace(99, np.nan)
bookings["past_no_shows"] = bookings["past_no_shows"].fillna(bookings["past_no_shows"].median())

bookings["months_member"] = bookings["months_member"].replace(999, np.nan)
bookings["months_member"] = bookings["months_member"].fillna(bookings["months_member"].median())

minutes = bookings["hours_booked_before"] > 168            # True for the rows saved in minutes
bookings.loc[minutes, "hours_booked_before"] = bookings.loc[minutes, "hours_booked_before"] / 60

bookings[["past_no_shows", "months_member", "hours_booked_before"]].describe().round(1)

In [ ]:
# Step 8: keep a copy of the table with the errors repaired, also for Part 3
repaired = bookings.copy()

No impossible values left. What is still extreme now is **real**. That is the interesting part.

## 2.1 · Members who often do not come: keep

**Example**

In [ ]:
# Step 1: picture of the earlier no-shows
bookings["past_no_shows"].plot(kind="box", vert=False, figsize=(7, 2), title="Earlier no-shows")
plt.show()

In [ ]:
# Step 2: count every value. sort_index() sorts by the value (0, 1, 2, ...), not by the count
bookings["past_no_shows"].value_counts().sort_index()

Most members missed 0 to 3 classes before. A few missed **8 to 19**. The boxplot shows them as outliers. Real people, but should we keep them?

**Your turn · 2.1 · Members who often do not come**

<details><summary>Hint</summary>

```python
8
bookings["often_absent"]
groupby("often_absent")
```

</details>

In [ ]:
# Step 1: a True/False column for members with 8 or more earlier no-shows (>= means "8 or more")
bookings["often_absent"] = bookings["past_no_shows"] >= ____
print(bookings["often_absent"].sum(), "bookings")

*You should see:* 75 bookings.

In [ ]:
# Step 2: first a picture: came and no-show, for these members and the others
# pd.crosstab counts came/no-show in each group; normalize="index" turns the counts into shares
table = pd.crosstab(bookings[____], bookings["no_show"], normalize="index")
table.columns = ["came", "no-show"]
table.plot(kind="barh", stacked=True, figsize=(6, 2), title="Came or not")
plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.show()

*You should see:* the orange part is much longer for True.

In [ ]:
# Step 3: the no-show rate
bookings.groupby(____)["no_show"].mean().round(2)

*You should see:* False 0.23, True **0.59**.

These members do not come in **59%** of their bookings, the others in 23%. They are rare, but they are **exactly the people we want to predict**. Removing or capping them would throw away the signal. **We keep them.**

The same is true for the founding members (120 to 180 months as member): real, and part of our members. **We keep them too.**

## 2.2 · Hours booked before: measure the impact
Many people book a few hours before a class, a few book days before (up to one week). All values are possible now. But do they distort our numbers? You do the same steps as with the repetitions in Part 1.

**Your turn · 2.2 · Mean and median**

<details><summary>Hint</summary>

```python
"hours_booked_before"
.median()
```

</details>

**Task:** Calculate the mean and the median of the hours. Write the **column name** in the gap.

In [ ]:
# Step 1: mean and median of the hours, as a small table
bookings[____].agg(["mean", "median"]).round(2)

*You should see:* mean 24.13, median 17.15.

**Task:** Draw the histogram with one line for the mean and one for the median. Fill in the **function for the median**.

In [ ]:
# Step 2: histogram with a red line for the mean and a black line for the median
bookings["hours_booked_before"].plot(kind="hist", bins=50, figsize=(8, 3), title="Hours booked before")
plt.axvline(bookings["hours_booked_before"].mean(), color="red", label="mean")
plt.axvline(bookings["hours_booked_before"].____(), color="black", label="median")
plt.legend()
plt.show()

*You should see:* a long tail to the right; the red mean line is to the right of the black median line.

The mean is **7 hours** bigger than the median. The few bookings made days before pull the mean to the right. If the gym says *"members book on average 24 hours before"*, that describes almost nobody: half of the bookings are made less than 17 hours before.

**Your turn · 2.3 · Find the outliers with the IQR**

<details><summary>Hint</summary>

```python
"hours_booked_before"
0.75
upper
"flag_hours_outlier"
```

</details>

**Task:** Show Q1 and Q3 of the hours in a table. Write the **column name** in the gap.

In [ ]:
# Step 1: Q1 (0.25) and Q3 (0.75)
bookings[____].quantile([0.25, 0.75])

*You should see:* 0.25 → 6.975 and 0.75 → 33.925.

**Task:** Save Q1 and Q3 in two variables and calculate the upper cut: Q3 + 1.5 × (Q3 − Q1). Fill in the **missing quantile**.

In [ ]:
# Step 2: the upper cut
# .quantile(0.25) -> Q1: 25% of the bookings are below this value
# .quantile(0.75) -> Q3: 75% of the bookings are below this value
q1 = bookings["hours_booked_before"].quantile(____)
q3 = bookings["hours_booked_before"].quantile(____)
upper = ____
print("upper cut:", upper, "hours")

*You should see:* 74.35 hours.

**Task:** Make a True/False column that marks every booking above the cut, and look at those bookings.

In [ ]:
# Step 3: flag the bookings above the cut
# bookings["hours_booked_before"] > ____   -> True if the booking was made more hours before than the cut
# bookings["flag_hours_outlier"] = ...     -> save True/False in a new column (at the end, on the right)
# bookings[bookings["flag_hours_outlier"]] -> show only the rows where the flag is True
bookings["flag_hours_outlier"] = bookings["hours_booked_before"] > ____
bookings[bookings["flag_hours_outlier"]].head()

*You should see:* bookings with 80 to 168 hours.

**Task:** Count the flagged bookings and compare their no-show rate with the others. Write the **flag column** in the gap.

In [ ]:
# Step 4: how many, and do they behave differently?
bookings.groupby(____)["no_show"].agg(["count", "mean"]).round(2)

*You should see:* False 1,434 bookings with 0.24, True **66** bookings with **0.33**.

The 66 early bookings do not come more often (33% against 24%). So they carry information: **we keep the rows**. But their extreme **values** pull the mean and, as you will see, the scale. So we **replace the values**, not the rows. Two techniques: **cap** and **log**.

**Your turn · 2.4 · Cap: every value above the cut becomes the cut**

<details><summary>Hint</summary>

```python
upper
"capped_hours"
```

</details>

**Task:** Make a new column where every value above `upper` becomes `upper`. `.clip(upper=x)` does it. Fill in the **cap**.

In [ ]:
# Step 1: the capped hours in a new column
# .clip(upper=____)                        -> every value above the cut becomes the cut, all others stay
# bookings["capped_hours"] = ...           -> save the result in a new column (at the end, on the right)
# bookings[bookings["flag_hours_outlier"]] -> show only the flagged rows, to see what changed
bookings["capped_hours"] = bookings["hours_booked_before"].clip(upper=____)
bookings[bookings["flag_hours_outlier"]].head()

*You should see:* the original hours (89.6, 80.0, 168.0, ...) next to the capped ones (all 74.35).

**Task:** Compare mean, median and max before and after the cap. Write the **new column** in the gap.

In [ ]:
# Step 2: before and after, side by side
# [["hours_booked_before", ____]]      -> the original column and the new capped column
# .agg(["mean", "median", "max"])      -> calculate these three numbers for each column
# .round(2)                            -> 2 decimals
bookings[["hours_booked_before", ____]].agg(["mean", "median", "max"]).round(2)

*You should see:* mean 24.13 → 23.17, median 17.15 → 17.15, max 168 → 74.35.

**Task:** Draw the boxplot before and after the cap. Write the **new column** in the gap.

In [ ]:
# Step 3: boxplot before and after the cap
# [["hours_booked_before", ____]]   -> the original column and the new capped column
# .plot(kind="box", vert=False)     -> one horizontal boxplot for each column
# figsize=(9, 3)                    -> width and height of the picture
bookings[["hours_booked_before", ____]].plot(kind="box", vert=False, figsize=(9, 3), title="Hours: before and after the cap")
plt.show()

*You should see:* the same box and median; after the cap the dots are gone and the whisker ends at 74.35.

**Your turn · 2.5 · Log: shorten the long tail**

<details><summary>Hint</summary>

```python
np.log1p
"capped_hours"
"log_hours"
```

</details>

**Task:** Make a new column with the log of the hours. `np.log1p` takes the log of (1 + value): it makes big numbers much smaller and also works with 0. Fill in the **function**.

In [ ]:
# Step 1: the log of the hours in a new column (at the end of the table)
# np.____(x)  -> log(1 + x): big numbers become much smaller, small numbers a bit smaller
#               3.3 hours -> 1.46, 61 hours -> 4.13, 168 hours -> 5.13
bookings["log_hours"] = np.____(bookings["hours_booked_before"])
bookings.head()

*You should see:* hours like 3.3 and 61.0 become 1.46 and 4.13.

**Task:** Draw the three versions next to each other. Write the **two new columns** in the gaps.

In [ ]:
# Step 2: original, capped and log
fig, ax = plt.subplots(1, 3, figsize=(13, 3))
bookings["hours_booked_before"].plot(kind="hist", bins=40, ax=ax[0], title="original")
bookings[____].plot(kind="hist", bins=40, ax=ax[1], title="capped")
bookings[____].plot(kind="hist", bins=40, ax=ax[2], title="log")
plt.show()

*You should see:* original: long tail; capped: the tail ends in one bar at 74; log: no long tail any more.

## 2.3 · The impact on the scale
Now the formula from Part 1: scaled = (value - min) / (max - min). We scale the three versions and look where three typical bookings land: 6, 17 and 34 hours before (Q1, median, Q3).

**Your turn · 2.6 · Scale the three versions**

<details><summary>Hint</summary>

```python
.min()
"capped_hours"
"log_hours"
```

</details>

**Task:** Scale the original hours with (value − min) / (max − min). Fill in the **function for the minimum**.

In [ ]:
# Step 1: scale the original hours to 0 ... 1
# x.min()                     -> the smallest value (0.5 hours)
# x.max()                     -> the biggest value (168 hours)
# (x - min) / (max - min)     -> the smallest becomes 0, the biggest becomes 1, all others in between
x = bookings["hours_booked_before"]
bookings["scaled_original"] = (x - x.____()) / (x.max() - x.min())
bookings.head()

*You should see:* 3.3 hours → 0.02, 61 hours → 0.36.

**Task:** The same formula for the capped hours. Write the **column name** in the gap.

In [ ]:
# Step 2: scale the capped hours, the same formula
# now the biggest value is the cap (74.35), so the typical bookings get bigger scaled values
x = bookings[____]
bookings["scaled_capped"] = (x - x.min()) / (x.max() - x.min())
bookings.head()

*You should see:* 3.3 hours → 0.04, 61 hours → 0.82.

**Task:** The same formula for the log hours. Write the **column name** in the gap.

In [ ]:
# Step 3: scale the log hours, the same formula
x = bookings[____]
bookings["scaled_log"] = (x - x.min()) / (x.max() - x.min())
bookings.head()

**Task:** Just run: the three scaled versions as histograms.

In [ ]:
# Step 4: picture
fig, ax = plt.subplots(1, 3, figsize=(13, 3), sharey=True)
for axis, column in zip(ax, ["scaled_original", "scaled_capped", "scaled_log"]):
    bookings[column].plot(kind="hist", bins=40, ax=axis, title=column)
    axis.set_xlim(0, 1)
plt.show()

*You should see:* original: almost everything on the left; capped and log: spread over the whole scale.

**Task:** Just run: where do Q1, the median and Q3 land in each version?

In [ ]:
# Step 5: Q1, median and Q3 of the three scaled columns
bookings[["scaled_original", "scaled_capped", "scaled_log"]].quantile([0.25, 0.5, 0.75]).round(2)

*You should see:* original 0.04, 0.10, 0.20; capped 0.09, 0.23, 0.45; log 0.35, 0.53, 0.67.

With the original hours, three quarters of the bookings are squeezed below **0.2**, because one booking was made a week before. After the cap or the log the bookings use the whole scale.

**Our decision for this column**

| | cap | log |
|---|---|---|
| keeps the rows | yes | yes |
| keeps every value different | no: 66 values become 74.35 | yes |
| easy to read | yes: still hours | no: what is 4.1? |
| fixes the mean and the scale | yes | yes |

Both are fine. We keep both columns and let Part 3 show what a model does with them.

**What we did in Part 2**

| Column | What | Error or real? | Treatment |
|---|---|---|---|
| past_no_shows | 99 | error | median |
| past_no_shows | 8 to 19 | real, **the signal** | **keep** |
| months_member | 999 | error | median |
| months_member | 120 to 180 | real | **keep** |
| hours_booked_before | saved in minutes | error | **repair** (/ 60) |
| hours_booked_before | 75 to 168 | real, but distorts mean and scale | **cap** or **log** |

## 2.7 · Compare and present
You treated the hours in two ways: **cap** and **log**. Which one is better? Knowing the steps is not enough. At the end of an analysis you must **show** what your treatment did. Always with three things:

1. **one table** that compares the versions,
2. **one picture**,
3. **one sentence** with your decision and the reason.

Here is a complete example. In Part 4 you do the same yourself.

**What is skewness?** It measures how lopsided a column is.
- **0** means symmetric: as many values on the left as on the right of the middle, like a bell.
- **Bigger than 0** means a long tail to the right: most values are small, a few are very big. The hours (1.85) and the income (32.96) look like this.
- **Smaller than 0** means a long tail to the left.

Why do we care? Many methods (the mean, a straight line, scaling, linear models) work best when a column is roughly symmetric. A long tail means a few values pull everything. Skewness has **no unit**, so we can compare the hours with the log hours, which the mean cannot do.

In [ ]:
# Step 1: a small function that describes one column with four numbers
# def summary(x):  -> we define our own function; x is the column we give it
# low, high        -> the IQR cuts of this column
# scaled           -> the column scaled to 0 ... 1
# .skew()          -> skewness: how lopsided the column is. 0 = symmetric, big = long tail to the right
#                     it has no unit, so we can compare hours with log hours
# pd.Series({...}) -> the four numbers with a name for each
def summary(x):
    q1 = x.quantile(0.25)
    q3 = x.quantile(0.75)
    low = q1 - 1.5 * (q3 - q1)
    high = q3 + 1.5 * (q3 - q1)
    scaled = (x - x.min()) / (x.max() - x.min())
    return pd.Series({
        "skewness": x.skew(),
        "median": x.median(),
        "outliers (IQR)": ((x < low) | (x > high)).sum(),
        "scaled median": scaled.median(),
    })

# test it on the original hours
summary(bookings["hours_booked_before"]).round(2)

In [ ]:
# Step 2: the comparison table: one row per version
# pd.DataFrame([...], index=[...]) -> put the three results under each other and give each row a name
comparison = pd.DataFrame([
    summary(bookings["hours_booked_before"]),
    summary(bookings["capped_hours"]),
    summary(bookings["log_hours"]),
], index=["no treatment", "cap", "log"])
comparison.round(2)

How to read the table:
- **skewness:** how lopsided the column is. The original hours have a long tail to the right (**1.85**). The cap makes it smaller (**1.04**), the log makes the column almost symmetric (**-0.38**).
- **median:** the cap does not change it (17.15 hours). The log changes the unit (log hours), so its median is not comparable with the others.
- **outliers (IQR):** 66 → **0** with the cap and with the log. The cap pulls the long values back, the log shortens the long tail.
- **scaled median:** the most important column for the next steps. Without treatment the typical booking lands at **0.10**, with the cap at **0.23**, with the log at **0.53**.

The QQ plot shows the same: the original and the capped hours bend away from the line, the log hours follow it much better.

In [ ]:
# Step 3: one picture: the three versions as boxplots, each with its own axis
fig, ax = plt.subplots(1, 3, figsize=(13, 3))
bookings["hours_booked_before"].plot(kind="box", vert=False, ax=ax[0], title="no treatment")
bookings["capped_hours"].plot(kind="box", vert=False, ax=ax[1], title="cap")
bookings["log_hours"].plot(kind="box", vert=False, ax=ax[2], title="log")
plt.tight_layout()
plt.show()

**How to read a QQ plot** (you know it from last semester): each blue point is one value of the column. The red line shows where the points would be if the column were a perfect bell shape (a normal distribution).
- **Points on the line:** the column is close to a bell shape.
- **Points bending up on the right:** a long tail to the right, the big values are bigger than a bell shape would allow.
- **A flat part:** many identical values, for example all the capped values at 74.35.

In [ ]:
# Step 3b: QQ plot, you know it from last semester: the closer the points are to the line, the more "normal" the column
# probplot(x, dist="norm", plot=ax) draws the points and the reference line
from scipy.stats import probplot
fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))
probplot(bookings["hours_booked_before"], dist="norm", plot=ax[0])
probplot(bookings["capped_hours"], dist="norm", plot=ax[1])
probplot(bookings["log_hours"], dist="norm", plot=ax[2])
ax[0].set_title("no treatment"); ax[1].set_title("cap"); ax[2].set_title("log")
plt.tight_layout()
plt.show()

- **No treatment:** the points bend up strongly on the right: the long tail.
- **Cap:** the curve stops in a flat line at 74.35, where all the capped values sit. The tail is cut, but the shape is still not a bell.
- **Log:** the points follow the line almost everywhere. The log turned the long tail into a nearly symmetric shape. That is why its skewness is close to 0.

**Why log1p and not log?** `np.log1p(x)` calculates log(1 + x). For big values it is almost the same as the log, but it also works when a value is 0 (the log of 0 does not exist). It is the same function you used last semester.

In [ ]:
# Step 4: does a model care? model_test (from the setup) trains a simple model with three inputs
# and counts the real no-shows among the 100 bookings with the highest risk. At random: about 24
# Only the hours column changes between the three calls
result_none = model_test(bookings, "no_show", ["past_no_shows", "months_member", "hours_booked_before"])
result_cap = model_test(bookings, "no_show", ["past_no_shows", "months_member", "capped_hours"])
result_log = model_test(bookings, "no_show", ["past_no_shows", "months_member", "log_hours"])
print("no treatment:", result_none, "  cap:", result_cap, "  log:", result_log)

In [ ]:
# Step 5: add the model result as a new column to the comparison table
comparison["no-shows in top 100"] = [result_none, result_cap, result_log]
comparison.round(2)

**Step 6: the sentence.**

> *"We keep all early bookings, because they do not come more often (33% against 24%). We treat the hours with the **log**: it keeps every value different and makes the column almost symmetric (skewness -0.38 instead of 1.85), leaves no outliers and gives the best spread on the 0 to 1 scale (median 0.53 instead of 0.10). In our simple model, the treatment changes the result only a little (42, 38 and 41 no-shows of 100), because the hours are a weak predictor of no-shows."*

That is a complete result: **table, picture, sentence**. The last part is honest: a treatment does not always make a big difference, and saying so is part of a good analysis.

---
# Part 3 · Does the treatment change a model? (just run)
A helper from the setup trains two models on three versions of the table: the **raw** export, the table with the **errors repaired**, and the table where the real outliers of the hours are also **treated** (log). Then it takes the 100 bookings with the highest risk in test data and counts how many are real no-shows. **At random**, about 25 of 100 would be no-shows. You will learn these models later; today just look at the numbers.

In [ ]:
look_ahead({"raw export": raw, "errors repaired": repaired, "real outliers treated": bookings})

- **Repairing the errors** helps the logistic regression: it finds 3 more no-shows (45 → 48). It multiplies every value by a weight, so a 99 or a 999 pulls it.
- **Treating the real outliers** of the hours changes nothing here (48 → 48). The early bookings were real, and this model and this column could handle them.
- The **decision tree** hardly changes at all. It only asks *"is the value above a border?"*, so it does not care how extreme a value is.

This is an honest result: errors are wrong for every method, so we always repair them. Treating **real** outliers helps some methods and some columns (you saw it with the line and the scale in Part 1), and does nothing for others. So: **measure before and after**, every time.

---
# Part 4 · One more example: real bank loans
The same steps on **real data**: 32,581 loans of a bank (credit risk dataset, Kaggle). One row = one loan. The target is `loan_status`: 1 = the person did **not** pay back (default), 0 = paid back.

> Put `credit_risk_dataset.csv` in the folder `data` next to this notebook.

**Example**

In [ ]:
# Step 1: load the file and remove duplicate rows
credit = pd.read_csv(find_data("credit_risk_dataset.csv")).drop_duplicates()
credit.shape

In [ ]:
# Step 2: the first rows
credit.head()

In [ ]:
# Step 3: three number columns: age, yearly income and years in the job
credit[["person_age", "person_income", "person_emp_length"]].describe().round(0)

- **Age** max **144**, **years in the job** max **123**: impossible. Errors.
- **Income** max **6,000,000** per year, while the median is 55,000. Possible: some people are rich. Real.

First the errors, the way you know.

**Example**

In [ ]:
# Step 4: the errors: age above 100 and more than 60 years in the job become empty
print("age above 100:", (credit["person_age"] > 100).sum(), "  job above 60 years:", (credit["person_emp_length"] > 60).sum())
credit.loc[credit["person_age"] > 100, "person_age"] = np.nan
credit.loc[credit["person_emp_length"] > 60, "person_emp_length"] = np.nan

Now the real outliers: the income. Same steps as with the hours in Part 2.

**Your turn · 4.1 · Mean and median of the income**

<details><summary>Hint</summary>

```python
"person_income"
"mean"
"median"
```

</details>

**Task:** Calculate the mean and the median of the income. Write the **column name** in the gap.

In [ ]:
# Step 1: mean and median, as a small table
# credit[____]                -> the column we study: the yearly income
# .agg(["____", "____"])      -> calculate these two numbers: the average and the middle value
# .round(0)                   -> no decimals
credit[____].agg(["____", "____"]).round(0)

*You should see:* mean 66,092, median 55,000.

**Task:** Just run: a boxplot of the income.

In [ ]:
# Step 2: boxplot
_______["__________"].plot(kind="box", vert=False, figsize=(8, 2), title="Yearly income")
plt.show()

*You should see:* the box is a thin line on the left, and dots up to 6 million.

**Your turn · 4.2 · Find the outliers and decide**

<details><summary>Hint</summary>

```python
0.75
upper
"flag_high_income"
```

</details>

**Task:** Calculate the upper cut of the income. Fill in the **missing quantile**.

In [ ]:
# Step 1: Q1, Q3 and the upper cut
q1 = ________["___________"]._________(____)
q3 = ________["___________"]._________(____)
upper = q_ + 1.5 * (q_ - q_)
print("upper cut:", upper)

*You should see:* 140,232.

**Task:** Make a True/False column for incomes above the cut and look at those people. Fill in the **cut**.

In [ ]:
# Step 2: flag the high incomes
# credit["person_income"] > ____    -> True if the income is above the cut
# credit["flag_high_income"] = ...  -> save True/False in a new column (at the end, on the right)

credit["flag_high_income"] = credit["person_income"] > ____
credit[credit["flag_high_income"]].head()

*You should see:* people with incomes of 150,000 and more.

**Task:** Count them and compare how often they do not pay back. Write the **flag column** in the gap.

In [ ]:
# Step 3: how many, and how often they default
credit.groupby(____)["loan_status"].agg(["count", "mean"]).round(2)

*You should see:* False 30,938 loans with 0.22, True **1,478** loans with **0.10**.

People with a high income default in **10%** of the loans, the others in 22%. A high income is **information** about the risk. Deleting these 1,478 people would be wrong. We keep the rows and treat the **values**.

**Your turn · 4.3 · Cap and log**

<details><summary>Hint</summary>

```python
"person_income"
upper
```

</details>

**Task:** Make a capped version of the income. Fill in the **column** you want to cap and the **cap**.

In [ ]:
# Step 1: cap the income at the upper cut
# credit[____].clip(upper=____)       -> every income above the cut becomes the cut, all others stay
# credit["capped_income"] = ...       -> save the result in a new column (at the end, on the right)
# credit[credit["flag_high_income"]]  -> show only the high incomes, to see what changed
credit["capped_income"] = credit[____].clip(upper=____)
credit[credit["flag_high_income"]].head()

*You should see:* the high incomes next to the capped ones (all 140,232).

**Task:** Make a log version of the income. Fill in the **column**.

In [ ]:
# Step 2: the log of the income, in a new column at the end of the table
# np.log1p(x)  -> log(1 + x): 59,000 -> 10.99, 6,000,000 -> 15.61
credit["log_income"] = np.log1p(credit[____])
credit.head()

*You should see:* 59,000 → 10.99, 9,600 → 9.17.

**Task:** Just run: the three versions side by side, as numbers and as boxplots.

In [ ]:
# Step 3: the three versions in one table
credit[["person_income", "capped_income", "log_income"]].agg(["mean", "median", "max"]).round(1)

*You should see:* mean 66,092 → 62,437 after the cap; max 6,000,000 → 140,232.

**Task:** Just run: original and capped as boxplots.

In [ ]:
# Step 4: boxplot before and after the cap
credit[["person_income", "capped_income"]].plot(kind="box", vert=False, figsize=(9, 3), title="Income: before and after the cap")
plt.show()

*You should see:* before: dots up to 6 million squeeze the box; after: a normal box that ends at 140,232.

**Your turn · 4.4 · The impact on the scale**

<details><summary>Hint</summary>

```python
"person_income"
"capped_income"
```

</details>

**Task:** Scale the original income with (value − min) / (max − min). Fill in the **column**.

In [ ]:
# Step 1: scale the original income to 0 ... 1
# x.min() is 4,000 and x.max() is 6,000,000: one rich person decides the whole scale
x = credit[____]
credit["scaled_original"] = (x - x.min()) / (x.max() - x.min())
credit.head()

*You should see:* 59,000 → 0.009, 9,600 → 0.001: almost 0.

**Task:** The same for the capped income. Fill in the **column**.

In [ ]:
# Step 2: scale the capped income, the same formula
# now the biggest value is the cap (140,232)
x = credit[____]
credit["scaled_capped"] = (x - x.min()) / (x.max() - x.min())
credit.head()

*You should see:* 59,000 → 0.40, 9,600 → 0.04.

**Task:** Just run: Q1, median and Q3 of both scaled versions.

In [ ]:
# Step 3: where do the typical people land?
credit[["scaled_original", "scaled_capped"]].quantile([0.25, 0.5, 0.75]).round(3)

*You should see:* original 0.006, 0.009, 0.013; capped 0.254, 0.374, 0.552.

With the original income, 75% of the people are squeezed below **0.013**: for a model, an income of 30,000 and one of 80,000 look almost the same, because one person earns 6 million. After the cap, the differences are visible again.

**Your turn · 4.5 · Compare and present**

Now you prove your treatment, the same way as in 2.7: one table, one picture, one sentence. The function `summary` and the helper `model_test` already exist.

<details><summary>Hint</summary>

```python
"person_income"
"log_income"
"capped_income"
result_none, result_cap, result_log
```

</details>

**Task:** Make the comparison table: one row for each version of the income. Write the **two missing columns** in the gaps.

In [ ]:
# Step 1: the comparison table
comparison_credit = pd.DataFrame([
    summary(credit[____]),
    summary(credit["capped_income"]),
    summary(credit[____]),
], index=["no treatment", "cap", "log"])
comparison_credit.round(2)

*You should see:* skewness 32.96 → 0.87 (cap) and 0.16 (log); outliers 1,478 → 0 with the cap, 571 with the log (on both sides of a bell shape); scaled median 0.01 → 0.37 (cap) and 0.36 (log).

**Task:** Draw the three versions as boxplots. Write the **column** for the cap in the gap.

In [ ]:
# Step 2: one picture
fig, ax = plt.subplots(1, 3, figsize=(13, 3))
credit["person_income"].plot(kind="box", vert=False, ax=ax[0], title="no treatment")
credit[____].plot(kind="box", vert=False, ax=ax[1], title="cap")
credit["log_income"].plot(kind="box", vert=False, ax=ax[2], title="log")
plt.tight_layout()
plt.show()

*You should see:* no treatment: a squeezed box and dots up to 6 million; cap: a normal box; log: a box in the middle with dots on both sides.

**Task:** Test a simple model with four inputs: the income, the loan amount, the interest rate and the age. Only the income changes between the three calls. Write the **income column** in each gap. Here we count the defaults among the **1,000** loans with the highest risk. At random: about 219.

In [ ]:
# Step 3: does a model care?
result_none = model_test(credit, "loan_status", [____, "loan_amnt", "loan_int_rate", "person_age"], top=1000)
result_cap = model_test(credit, "loan_status", [____, "loan_amnt", "loan_int_rate", "person_age"], top=1000)
result_log = model_test(credit, "loan_status", [____, "loan_amnt", "loan_int_rate", "person_age"], top=1000)
print("no treatment:", result_none, "  cap:", result_cap, "  log:", result_log)

*You should see:* no treatment 646, cap 694, log **721**.

**Task:** Add the model results as a new column to your table. Write the **three results** in the list.

In [ ]:
# Step 4: the final table
comparison_credit["defaults in top 1000"] = [____, ____, ____]
comparison_credit.round(2)

*You should see:* the table from Step 1 with a new column on the right: 646, 694, 721.

Here the treatment makes a **big** difference: with the log the model finds **75 more defaults** than without treatment. With the original income, the person who earns 6 million squeezes everybody else close to 0 on the scale, and the model can hardly use the income.

**Your decision:** which treatment would you choose for the income, keep, cap or log? Write one or two sentences, like the example in 2.7, and give a reason with numbers from your table (think of the 10% default rate of the high incomes, and of what is easy to explain to the bank).

*Your answer:* 

---
## Summary
| What we found | Error or real? | What we did |
|---|---|---|
| 1300 kg, 120 reps, 30 sets | error, typo | **repair** |
| 0 kg, 0 reps | error, true value unknown | empty |
| 57 kg | flagged, but judged possible (add-on plate) | **keep** |
| 1 to 3 reps, 20 to 47 reps | real, rare | distort mean, scale and prediction: **cap**, or keep and model them separately |
| leg press "outliers" | real | compare each machine with itself |
| 99 earlier no-shows, 999 months | error | median |
| 8 to 19 earlier no-shows | real, **the signal** | **keep** |
| early bookings (75 to 168 h) | real | **cap** or **log** |
| age 144, 123 years in the job | error | empty |
| incomes up to 6 million | real, **information** about the risk | keep the rows, **cap** or **log** the values |

**Errors: repair them. Real outliers: measure what they do to the mean, the scale and the prediction, then decide: keep, cap or log.**

---
## Answers
Try first, then compare. Click to open.

<details><summary>2.1 · Members who often do not come</summary>

```python
# Step 1: a True/False column for members with 8 or more earlier no-shows (>= means "8 or more")
bookings["often_absent"] = bookings["past_no_shows"] >= 8
print(bookings["often_absent"].sum(), "bookings")

# Step 2: first a picture: came and no-show, for these members and the others
# pd.crosstab counts came/no-show in each group; normalize="index" turns the counts into shares
table = pd.crosstab(bookings["often_absent"], bookings["no_show"], normalize="index")
table.columns = ["came", "no-show"]
table.plot(kind="barh", stacked=True, figsize=(6, 2), title="Came or not")
plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.show()

# Step 3: the no-show rate
bookings.groupby("often_absent")["no_show"].mean().round(2)
```

</details>

<details><summary>2.2 · Mean and median</summary>

```python
# Step 1: mean and median of the hours, as a small table
bookings["hours_booked_before"].agg(["mean", "median"]).round(2)

# Step 2: histogram with a red line for the mean and a black line for the median
bookings["hours_booked_before"].plot(kind="hist", bins=50, figsize=(8, 3), title="Hours booked before")
plt.axvline(bookings["hours_booked_before"].mean(), color="red", label="mean")
plt.axvline(bookings["hours_booked_before"].median(), color="black", label="median")
plt.legend()
plt.show()
```

</details>

<details><summary>2.3 · Find the outliers with the IQR</summary>

```python
# Step 1: Q1 (0.25) and Q3 (0.75)
bookings["hours_booked_before"].quantile([0.25, 0.75])

# Step 2: the upper cut
q1 = bookings["hours_booked_before"].quantile(0.25)
q3 = bookings["hours_booked_before"].quantile(0.75)
upper = q3 + 1.5 * (q3 - q1)
print("upper cut:", upper, "hours")

# Step 3: flag the bookings above the cut
bookings["flag_hours_outlier"] = bookings["hours_booked_before"] > upper
bookings[bookings["flag_hours_outlier"]].head()

# Step 4: how many, and do they behave differently?
bookings.groupby("flag_hours_outlier")["no_show"].agg(["count", "mean"]).round(2)
```

</details>

<details><summary>2.4 · Cap: every value above the cut becomes the cut</summary>

```python
# Step 1: the capped hours in a new column, next to the original for the flagged rows
bookings["capped_hours"] = bookings["hours_booked_before"].clip(upper=upper)
bookings[bookings["flag_hours_outlier"]].head()

# Step 2: before and after, side by side
bookings[["hours_booked_before", "capped_hours"]].agg(["mean", "median", "max"]).round(2)

# Step 3: boxplot before and after the cap
bookings[["hours_booked_before", "capped_hours"]].plot(kind="box", vert=False, figsize=(9, 3), title="Hours: before and after the cap")
plt.show()
```

</details>

<details><summary>2.5 · Log: shorten the long tail</summary>

```python
# Step 1: the log of the hours in a new column
bookings["log_hours"] = np.log1p(bookings["hours_booked_before"])
bookings.head()

# Step 2: original, capped and log
fig, ax = plt.subplots(1, 3, figsize=(13, 3))
bookings["hours_booked_before"].plot(kind="hist", bins=40, ax=ax[0], title="original")
bookings["capped_hours"].plot(kind="hist", bins=40, ax=ax[1], title="capped")
bookings["log_hours"].plot(kind="hist", bins=40, ax=ax[2], title="log")
plt.show()
```

</details>

<details><summary>2.6 · Scale the three versions</summary>

```python
# Step 1: scale the original hours
x = bookings["hours_booked_before"]
bookings["scaled_original"] = (x - x.min()) / (x.max() - x.min())
bookings.head()

# Step 2: scale the capped hours
x = bookings["capped_hours"]
bookings["scaled_capped"] = (x - x.min()) / (x.max() - x.min())
bookings.head()

# Step 3: scale the log hours
x = bookings["log_hours"]
bookings["scaled_log"] = (x - x.min()) / (x.max() - x.min())
bookings.head()

# Step 4: picture
fig, ax = plt.subplots(1, 3, figsize=(13, 3), sharey=True)
for axis, column in zip(ax, ["scaled_original", "scaled_capped", "scaled_log"]):
    bookings[column].plot(kind="hist", bins=40, ax=axis, title=column)
    axis.set_xlim(0, 1)
plt.show()

# Step 5: Q1, median and Q3 of the three scaled columns
bookings[["scaled_original", "scaled_capped", "scaled_log"]].quantile([0.25, 0.5, 0.75]).round(2)
```

</details>

<details><summary>4.1 · Mean and median of the income</summary>

```python
# Step 1: mean and median, as a small table
credit["person_income"].agg(["mean", "median"]).round(0)

# Step 2: boxplot
credit["person_income"].plot(kind="box", vert=False, figsize=(8, 2), title="Yearly income")
plt.show()
```

</details>

<details><summary>4.2 · Find the outliers and decide</summary>

```python
# Step 1: Q1, Q3 and the upper cut
q1 = credit["person_income"].quantile(0.25)
q3 = credit["person_income"].quantile(0.75)
upper = q3 + 1.5 * (q3 - q1)
print("upper cut:", upper)

# Step 2: flag the high incomes
credit["flag_high_income"] = credit["person_income"] > upper
credit[credit["flag_high_income"]].head()

# Step 3: how many, and how often they default
credit.groupby("flag_high_income")["loan_status"].agg(["count", "mean"]).round(2)
```

</details>

<details><summary>4.3 · Cap and log</summary>

```python
# Step 1: cap the income at the upper cut
credit["capped_income"] = credit["person_income"].clip(upper=upper)
credit[credit["flag_high_income"]].head()

# Step 2: the log of the income
credit["log_income"] = np.log1p(credit["person_income"])
credit.head()

# Step 3: the three versions in one table
credit[["person_income", "capped_income", "log_income"]].agg(["mean", "median", "max"]).round(1)

# Step 4: boxplot before and after the cap
credit[["person_income", "capped_income"]].plot(kind="box", vert=False, figsize=(9, 3), title="Income: before and after the cap")
plt.show()
```

</details>

<details><summary>4.4 · The impact on the scale</summary>

```python
# Step 1: scale the original income
x = credit["person_income"]
credit["scaled_original"] = (x - x.min()) / (x.max() - x.min())
credit.head()

# Step 2: scale the capped income
x = credit["capped_income"]
credit["scaled_capped"] = (x - x.min()) / (x.max() - x.min())
credit.head()

# Step 3: where do the typical people land?
credit[["scaled_original", "scaled_capped"]].quantile([0.25, 0.5, 0.75]).round(3)
```

</details>

<details><summary>4.5 · Compare and present</summary>

```python
# Step 1: the comparison table
comparison_credit = pd.DataFrame([
    summary(credit["person_income"]),
    summary(credit["capped_income"]),
    summary(credit["log_income"]),
], index=["no treatment", "cap", "log"])
comparison_credit.round(2)

# Step 2: one picture
fig, ax = plt.subplots(1, 3, figsize=(13, 3))
credit["person_income"].plot(kind="box", vert=False, ax=ax[0], title="no treatment")
credit["capped_income"].plot(kind="box", vert=False, ax=ax[1], title="cap")
credit["log_income"].plot(kind="box", vert=False, ax=ax[2], title="log")
plt.tight_layout()
plt.show()

# Step 3: does a model care?
result_none = model_test(credit, "loan_status", ["person_income", "loan_amnt", "loan_int_rate", "person_age"], top=1000)
result_cap = model_test(credit, "loan_status", ["capped_income", "loan_amnt", "loan_int_rate", "person_age"], top=1000)
result_log = model_test(credit, "loan_status", ["log_income", "loan_amnt", "loan_int_rate", "person_age"], top=1000)
print("no treatment:", result_none, "  cap:", result_cap, "  log:", result_log)

# Step 4: the final table
comparison_credit["defaults in top 1000"] = [result_none, result_cap, result_log]
comparison_credit.round(2)
```

</details>